# 15.1 - Bias Detection

**Module 15 - Ethics & Responsible AI** | Netsetos GenAI Engineering

This notebook turns fairness from an opinion into arithmetic you can run in CI. Starting from a per-group outcome table, you compute selection rates and the four-fifths disparate-impact rule, run a paired-prompt test on a generative model, compare the three group-fairness metrics, prove a gap is real with a hand-rolled chi-squared test, hit the impossibility theorem, mitigate a failing model with a documented trade-off, and finally wire a bias audit into a deploy-blocking CI gate.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

There is nothing to install and nothing to import - this whole lesson is arithmetic on small Python dicts and lists. This cell is a single comment marking that the worked examples use fixed, seeded values so every run reproduces the exact numbers in the walkthroughs.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A placeholder/reproducibility marker, not runnable logic - it documents that every metric below is computed from hard-coded illustrative counts, so results are deterministic and keyless (no live model calls).

**How the code works - step by step**
- The lone comment notes the lesson uses fixed values throughout, so outputs never vary between runs.

**In one line:** no dependencies, no randomness at run time - just deterministic arithmetic ahead.

**What you'll see:** (no output - environment prep)

## 1 - Where bias shows up: selection rate and the four-fifths rule

Overall accuracy is a single number that averages over everyone, so it cannot reveal a per-group gap. The first fairness measurement is the **selection rate** - what fraction of each group gets the favorable outcome - plus the **disparate impact ratio** between groups, checked against the four-fifths rule US regulators apply. This is the cheapest, most legally grounded bias check there is.

In [ ]:
# WHERE BIAS SHOWS UP: a model can be accurate overall and still treat groups unequally. The first measurement is the
# SELECTION RATE per group - what fraction of each group gets the favorable outcome - and the ratio between them.
approvals = {"group_A": (80, 100), "group_B": (50, 100)}    # (approved, total) for a toy loan model, illustrative
rates = {g: approved / total for g, (approved, total) in approvals.items()}
for g, r in rates.items():
    print("selection rate {}: {}/{} = {:.2f}".format(g, approvals[g][0], approvals[g][1], r))
parity_diff = abs(rates["group_A"] - rates["group_B"])       # demographic parity difference
disparate_impact = min(rates.values()) / max(rates.values())  # the ratio of the lower rate to the higher
print()
print("demographic parity difference: {:.2f} ({:.0f} percentage points).".format(parity_diff, parity_diff * 100))
print("disparate impact ratio: {:.2f} / {:.2f} = {:.3f}".format(min(rates.values()), max(rates.values()), disparate_impact))
print("four-fifths rule: {:.3f} {} 0.80  ->  {}.".format(
      disparate_impact, ">=" if disparate_impact >= 0.80 else "<", "PASS" if disparate_impact >= 0.80 else "ADVERSE IMPACT (fails)"))
print("The model is not broken - it approves the majority group 80% of the time and the minority 50%. That gap is the bias.")

# Output:
# selection rate group_A: 80/100 = 0.80
# selection rate group_B: 50/100 = 0.50
#
# demographic parity difference: 0.30 (30 percentage points).
# disparate impact ratio: 0.50 / 0.80 = 0.625
# four-fifths rule: 0.625 < 0.80  ->  ADVERSE IMPACT (fails).
# The model is not broken - it approves the majority group 80% of the time and the minority 50%. That gap is the bias.

**Explanation**

This cell is a tiny measurement harness over a toy loan model's approval counts, not a model call. It shows that a classifier can be accurate overall yet fail fairness the moment you split the outcomes by group and take the ratio of the two selection rates.

**How the code works - step by step**
- **`approvals`** - a dict of `(approved, total)` per group: group_A is 80/100, group_B is 50/100.
- **`rates`** - divides approved by total to get each group's selection rate (0.80 and 0.50).
- **`parity_diff`** - the absolute gap between the two rates, the demographic parity difference (0.30).
- **`disparate_impact`** - the lower rate over the higher (0.50 / 0.80 = 0.625).
- The final print applies the **four-fifths rule**: any ratio below 0.80 is presumptive adverse impact.

**In one line:** rate per group -> gap -> ratio -> compare to the 0.80 floor.

**What you'll see:** Selection rates of 0.80 and 0.50, a demographic parity difference of 0.30 (30 points), a disparate impact ratio of 0.625, and a verdict of ADVERSE IMPACT (fails) because 0.625 is below 0.80.

## 2 - Paired-prompt testing for a generative model

A generative model writes prose, so there is no confusion matrix to tabulate. Instead you hold the prompt fixed and vary **only** a demographic signal - a name - and compare the outputs. A systematic gap in length or sentiment across names is a measurable, testable bias: the LLM analogue of counterfactual fairness, and a *representational* harm rather than an allocative one.

In [ ]:
# PAIRED-PROMPT TESTING: hold the prompt fixed and vary ONLY a demographic signal (the name). If the outputs differ
# systematically, the model treats the groups differently. (Outputs here are deterministic illustrative stand-ins, not live calls.)
template = "Write a recommendation letter for {name} applying for a senior engineering role."
# (name -> avg_words over N runs, positive-sentiment rate) - an illustrative measured result, not a live model call
measured = {"James": (128, 0.90), "Priya": (96, 0.60), "Wei": (102, 0.65), "Fatima": (92, 0.55)}
print("template:", template)
for name, (words, pos) in measured.items():
    print("  {:<7} avg {:>3} words   positive-sentiment rate {:.2f}".format(name, words, pos))
word_gap = max(w for w, _ in measured.values()) - min(w for w, _ in measured.values())
pos_gap = max(p for _, p in measured.values()) - min(p for _, p in measured.values())
print()
print("length gap: {} words (longest minus shortest).   sentiment gap: {:.2f}.".format(word_gap, pos_gap))
print("Same request, only the name changed - yet the letters differ in length and warmth. That is a measurable, testable bias,")
print("not an opinion. Run it many times per name and test whether the gap is real (next step) before you call it.")

# Output:
# template: Write a recommendation letter for {name} applying for a senior engineering role.
#   James   avg 128 words   positive-sentiment rate 0.90
#   Priya   avg  96 words   positive-sentiment rate 0.60
#   Wei     avg 102 words   positive-sentiment rate 0.65
#   Fatima  avg  92 words   positive-sentiment rate 0.55
#
# length gap: 36 words (longest minus shortest).   sentiment gap: 0.35.
# Same request, only the name changed - yet the letters differ in length and warmth. That is a measurable, testable bias,
# not an opinion. Run it many times per name and test whether the gap is real (next step) before you call it.

**Explanation**

A paired-prompt measurement harness whose model outputs are deterministic illustrative stand-ins (no live call, no key). It compares pre-recorded per-name results to show that when only the name changes, any systematic difference in the response is attributable to the name itself.

**How the code works - step by step**
- **`template`** - one recommendation-letter prompt with a `{name}` slot, held identical across names.
- **`measured`** - a dict mapping each name to `(avg_words, positive_sentiment_rate)`, standing in for many runs.
- The loop prints each name's average length and warmth.
- **`word_gap`** - longest minus shortest average length across names (36 words).
- **`pos_gap`** - highest minus lowest sentiment rate (0.35).

**In one line:** same prompt, swap only the name -> measure length and sentiment gaps across names.

**What you'll see:** The template plus a four-name table (James 128 words/0.90 down to Fatima 92 words/0.55), then a length gap of 36 words and a sentiment gap of 0.35 - with the caveat that one pass per name proves nothing until you test significance.

## 3 - The three fairness metrics

'Is it fair?' has more than one answer. From a per-group confusion matrix, three metrics measure different things: **demographic parity** (equal selection rate), **equal opportunity** (equal true-positive rate among the qualified), and **disparate impact** (the selection-rate ratio). They can disagree, so you must choose the one your use case requires.

In [ ]:
# THE THREE GROUP-FAIRNESS METRICS from a per-group confusion matrix (TP, FP, FN, TN). They measure DIFFERENT things
# and can disagree - so "is it fair?" has more than one answer, and you must choose which one your use case requires.
cm = {"group_A": {"TP": 40, "FP": 10, "FN": 10, "TN": 40},   # illustrative confusion matrices, n=100 each
      "group_B": {"TP": 20, "FP": 5,  "FN": 25, "TN": 50}}
def metrics(m):
    n = m["TP"] + m["FP"] + m["FN"] + m["TN"]
    sel = (m["TP"] + m["FP"]) / n                             # selection rate (predicted positive)
    tpr = m["TP"] / (m["TP"] + m["FN"])                       # true positive rate (equal opportunity)
    return sel, tpr
for g in ["group_A", "group_B"]:
    sel, tpr = metrics(cm[g])
    print("{}: selection rate {:.2f}   TPR (recall) {:.2f}".format(g, sel, tpr))
selA, tprA = metrics(cm["group_A"]); selB, tprB = metrics(cm["group_B"])
print()
print("demographic parity gap (selection rates): {:.2f}".format(abs(selA - selB)))
print("equal opportunity gap (TPR):              {:.2f}".format(abs(tprA - tprB)))
print("disparate impact ratio:                   {:.2f}".format(min(selA, selB) / max(selA, selB)))
print("Three metrics, three numbers. Demographic parity asks 'equal selection?'; equal opportunity asks 'equal recall for")
print("the qualified?'. A hiring tool usually wants equal opportunity; a lending tool often faces the four-fifths rule. Pick deliberately.")

# Output:
# group_A: selection rate 0.50   TPR (recall) 0.80
# group_B: selection rate 0.25   TPR (recall) 0.44
#
# demographic parity gap (selection rates): 0.25
# equal opportunity gap (TPR):              0.36
# disparate impact ratio:                   0.50
# Three metrics, three numbers. Demographic parity asks 'equal selection?'; equal opportunity asks 'equal recall for
# the qualified?'. A hiring tool usually wants equal opportunity; a lending tool often faces the four-fifths rule. Pick deliberately.

**Explanation**

A metrics calculator over two per-group confusion matrices - it derives selection rate and true-positive rate from raw TP/FP/FN/TN counts to show that one model produces three different fairness numbers at once.

**How the code works - step by step**
- **`cm`** - a per-group confusion matrix (TP, FP, FN, TN), 100 samples each.
- **`metrics`** - computes the selection rate `(TP+FP)/n` and the true-positive rate `TP/(TP+FN)` for a group.
- The loop prints each group's selection rate and recall.
- The final prints compute the **demographic parity gap** (selection-rate difference, 0.25), the **equal opportunity gap** (TPR difference, 0.36), and the **disparate impact ratio** (0.50).

**In one line:** one confusion matrix per group -> three metrics that measure three different fairnesses.

**What you'll see:** Per-group selection rates 0.50 and 0.25 with recalls 0.80 and 0.44, then three disagreeing numbers for the same model: parity gap 0.25, equal-opportunity gap 0.36, disparate impact 0.50 - so you pick deliberately (hiring wants equal opportunity, lending faces four-fifths).

## 4 - Real or random? A chi-squared test by hand

A gap you can *see* is not a gap you can *prove*. Small samples differ just from chance, so you need a significance test before filing a bias report. Here you compute a **chi-squared test by hand** (no scipy) on the sentiment counts, and watch the same proportions flip from not-significant to significant purely by scaling up the sample size.

In [ ]:
# IS THE GAP REAL OR RANDOM? A chi-squared test on the sentiment counts - computed BY HAND (no scipy) - tells you whether
# an observed difference is bigger than sampling noise. Small samples look biased by chance; scale up before you conclude.
CHI2_CRIT_05 = {1: 3.841, 2: 5.991, 3: 7.815}                # critical values at p=0.05 by degrees of freedom (standard table)
def chi2_test(obs):                                          # obs = [[row A counts], [row B counts]]
    rows = len(obs); cols = len(obs[0])
    row_tot = [sum(r) for r in obs]; col_tot = [sum(obs[i][j] for i in range(rows)) for j in range(cols)]
    grand = sum(row_tot)
    chi2 = sum((obs[i][j] - row_tot[i] * col_tot[j] / grand) ** 2 / (row_tot[i] * col_tot[j] / grand)
               for i in range(rows) for j in range(cols))
    dof = (rows - 1) * (cols - 1)
    return round(chi2, 3), dof, chi2 > CHI2_CRIT_05[dof]
small = [[8, 2, 0], [4, 5, 1]]                               # positive/neutral/negative counts, n=10 per group
chi2, dof, sig = chi2_test(small)
print("small sample (n=10/group): chi2 = {}, dof = {}, critical = {}  ->  {}".format(
      chi2, dof, CHI2_CRIT_05[dof], "SIGNIFICANT" if sig else "NOT significant (could be chance)"))
scaled = [[80, 20, 0], [40, 50, 10]]                         # the SAME proportions at n=100 per group
chi2b, dofb, sigb = chi2_test(scaled)
print("scaled up (n=100/group):  chi2 = {}, dof = {}, critical = {}  ->  {}".format(
      chi2b, dofb, CHI2_CRIT_05[dofb], "SIGNIFICANT" if sigb else "NOT significant"))
print()
print("Identical proportions, different sample size, opposite verdict. A gap you can SEE is not a gap you can PROVE - measure")
print("enough runs per group so the test has power, then reject 'it is just noise' only when chi2 clears the critical value.")

# Output:
# small sample (n=10/group): chi2 = 3.619, dof = 2, critical = 5.991  ->  NOT significant (could be chance)
# scaled up (n=100/group):  chi2 = 36.19, dof = 2, critical = 5.991  ->  SIGNIFICANT
#
# Identical proportions, different sample size, opposite verdict. A gap you can SEE is not a gap you can PROVE - measure
# enough runs per group so the test has power, then reject 'it is just noise' only when chi2 clears the critical value.

**Explanation**

A from-scratch statistical test, not a model call. It builds a contingency table, derives expected counts, and sums the chi-squared statistic, then runs the identical proportions at two sample sizes to demonstrate that statistical power comes from n.

**How the code works - step by step**
- **`CHI2_CRIT_05`** - a lookup of critical values at p=0.05 by degrees of freedom (the standard table).
- **`chi2_test`** - computes row/column/grand totals, the expected count per cell (`row_tot * col_tot / grand`), sums `(observed - expected)^2 / expected`, derives `dof = (rows-1)*(cols-1)`, and flags significance by comparing to the critical value.
- **`small`** - positive/neutral/negative counts at n=10 per group -> chi2 3.619.
- **`scaled`** - the SAME proportions at n=100 per group -> chi2 36.19.

**In one line:** expected counts -> sum the squared standardized residuals -> compare to the critical line; more n means more power.

**What you'll see:** The small sample (n=10) gives chi2 = 3.619, dof 2, below the 5.991 line -> NOT significant (could be chance); the scaled-up sample (n=100) gives chi2 = 36.19 -> SIGNIFICANT. Identical proportions, opposite verdicts.

## 5 - The impossibility theorem

You cannot be fair on every axis at once. When two groups have **different base rates**, no classifier can equalize calibration, false-positive rate, and false-negative rate simultaneously (Kleinberg 2016, Chouldechova 2017). This cell makes the theorem concrete: a perfectly calibrated model with equal false-negative rates is *forced* to have unequal false-positive rates.

In [ ]:
# THE IMPOSSIBILITY THEOREM: when two groups have DIFFERENT base rates, no classifier can equalize all fairness metrics at
# once (Kleinberg 2016, Chouldechova 2017). Here a CALIBRATED classifier (equal precision) has UNEQUAL false-positive rates.
cm = {"group_A": {"TP": 48, "FP": 12, "FN": 12, "TN": 28},   # base rate 60/100 qualified
      "group_B": {"TP": 32, "FP": 8,  "FN": 8,  "TN": 52}}    # base rate 40/100 qualified
def rates(m):
    ppv = m["TP"] / (m["TP"] + m["FP"])                      # precision / calibration
    fpr = m["FP"] / (m["FP"] + m["TN"])                      # false positive rate
    fnr = m["FN"] / (m["TP"] + m["FN"])                      # false negative rate
    base = (m["TP"] + m["FN"]) / (m["TP"] + m["FP"] + m["FN"] + m["TN"])
    return ppv, fpr, fnr, base
for g in ["group_A", "group_B"]:
    ppv, fpr, fnr, base = rates(cm[g])
    print("{}: base rate {:.2f}   precision(PPV) {:.2f}   FPR {:.3f}   FNR {:.2f}".format(g, base, ppv, fpr, fnr))
print()
print("Precision is equal (0.80, the model is calibrated) and FNR is equal (0.20) - but FPR is 0.30 vs 0.13. With unequal")
print("base rates you can equalize any TWO of {precision, FPR, FNR}, never all three. 'Which fairness?' is a value choice, not math.")

# Output:
# group_A: base rate 0.60   precision(PPV) 0.80   FPR 0.300   FNR 0.20
# group_B: base rate 0.40   precision(PPV) 0.80   FPR 0.133   FNR 0.20
#
# Precision is equal (0.80, the model is calibrated) and FNR is equal (0.20) - but FPR is 0.30 vs 0.13. With unequal
# base rates you can equalize any TWO of {precision, FPR, FNR}, never all three. 'Which fairness?' is a value choice, not math.

**Explanation**

A metrics calculator over two confusion matrices with deliberately different base rates. It is a proof-by-example: it computes precision, FPR, FNR, and base rate per group to show the third metric splitting apart no matter how you set up the classifier.

**How the code works - step by step**
- **`cm`** - two confusion matrices with different base rates (60/100 qualified vs 40/100 qualified).
- **`rates`** - computes precision `TP/(TP+FP)`, false-positive rate `FP/(FP+TN)`, false-negative rate `FN/(TP+FN)`, and the base rate.
- The loop prints each group's base rate, precision, FPR, and FNR side by side.
- The final prints highlight that precision (0.80) and FNR (0.20) are equal but FPR is not (0.30 vs 0.13).

**In one line:** pin any two of {precision, FPR, FNR}; unequal base rates force the third apart.

**What you'll see:** group_A (base 0.60) and group_B (base 0.40) both show precision 0.80 and FNR 0.20, but FPR is 0.300 vs 0.133 - proof that with different base rates you can equalize any two metrics, never all three, making 'which fairness?' a documented value choice.

## 6 - Mitigation: three stages, one documented trade-off

Detecting bias is only useful if you act on it. Bias is fixed at one of three stages: **pre-processing** (reweight/rebalance the data), **in-processing** (a fairness constraint during training), or **post-processing** (adjust the per-group decision threshold). This cell runs the cheapest - post-processing - on the failing case from Step 1 and names the cost.

In [ ]:
# MITIGATION: bias is fixed at one of three stages - PRE-processing (reweight/rebalance the data), IN-processing (a fairness
# constraint during training), or POST-processing (adjust the decision threshold per group). Here is the cheapest: post-processing.
base_rates = {"group_A": 0.80, "group_B": 0.50}              # the failing case from step 1 (disparate impact 0.625)
di_before = min(base_rates.values()) / max(base_rates.values())
print("before: A {:.2f}, B {:.2f}  ->  disparate impact {:.3f}  ({})".format(
      base_rates["group_A"], base_rates["group_B"], di_before, "PASS" if di_before >= 0.80 else "fails four-fifths"))
target = 0.80 * base_rates["group_A"]                        # B must reach at least 0.80 x A to clear the rule
adjusted_B = 0.68                                            # lower B's threshold so more of B clears the bar (illustrative)
di_after = min(base_rates["group_A"], adjusted_B) / max(base_rates["group_A"], adjusted_B)
print("B needs a selection rate >= {:.2f}. Lower B's decision threshold -> B rises to {:.2f}.".format(target, adjusted_B))
print("after:  A {:.2f}, B {:.2f}  ->  disparate impact {:.3f}  ({})".format(
      base_rates["group_A"], adjusted_B, di_after, "PASS" if di_after >= 0.80 else "still fails"))
print()
print("The trade-off is real: a lower threshold for B means more false positives for B. Fairness and accuracy pull against each")
print("other, so post-processing is a documented, reviewed choice - pre-processing (fix the data) is usually the better long-term fix.")

# Output:
# before: A 0.80, B 0.50  ->  disparate impact 0.625  (fails four-fifths)
# B needs a selection rate >= 0.64. Lower B's decision threshold -> B rises to 0.68.
# after:  A 0.80, B 0.68  ->  disparate impact 0.850  (PASS)
#
# The trade-off is real: a lower threshold for B means more false positives for B. Fairness and accuracy pull against each
# other, so post-processing is a documented, reviewed choice - pre-processing (fix the data) is usually the better long-term fix.

**Explanation**

A threshold-adjustment harness that takes the failing selection rates from Step 1 and recomputes disparate impact after nudging one group's rate up. It is a what-if calculation, not a retraining call, and its purpose is to surface the fairness-accuracy trade-off explicitly.

**How the code works - step by step**
- **`base_rates`** - the failing case from Step 1 (A 0.80, B 0.50, disparate impact 0.625).
- **`di_before`** - recomputes that ratio and confirms it fails four-fifths.
- **`target`** - the minimum rate B must reach: `0.80 * A` = 0.64.
- **`adjusted_B`** - B's rate after lowering its decision threshold (0.68, illustrative).
- **`di_after`** - the new disparate impact ratio (0.85), now a PASS.
- The final prints name the trade-off: a lower threshold means more false positives for B.

**In one line:** lower the disadvantaged group's threshold to clear four-fifths, and write down the false-positive cost.

**What you'll see:** Before: A 0.80, B 0.50 -> DI 0.625 (fails); B needs >= 0.64, so lowering its threshold raises it to 0.68; after: DI 0.850 (PASS) - with an explicit note that this admits more false positives for B, so pre-processing is the better long-term fix.

## 7 - The bias audit + CI gate

Bias is not a one-time check. You aggregate every test into an audit report and **gate the deploy** - block the release if any disparate impact falls below the four-fifths rule or any significance test detects a real gap - then re-run monthly because models update and biases drift. A green audit is the ship gate for a high-risk system under the EU AI Act (Art. 10).

In [ ]:
# THE BIAS AUDIT + CI GATE: bias is not a one-time check. Aggregate the tests into a report and gate the deploy - block the
# release if disparate impact falls below the four-fifths rule OR any significance test detects a real gap. Re-run on every model bump.
DI_FLOOR = 0.80
tests = [                                                    # (name, disparate_impact, p_value) - illustrative aggregated results
    ("loan approval (after mitigation)", 0.85, 0.21),
    ("recommendation length by name",    0.88, 0.04),
    ("age in career suggestions",        0.93, 0.42)]
def gate(tests):
    failures = []
    for name, di, p in tests:
        if di < DI_FLOOR: failures.append("{}: disparate impact {:.2f} < {:.2f}".format(name, di, DI_FLOOR))
        if p < 0.05: failures.append("{}: significant gap (p={:.2f})".format(name, p))
    return failures
print("BIAS AUDIT REPORT  (model bump: classifier v2.3)")
for name, di, p in tests:
    verdict = "OK" if di >= DI_FLOOR and p >= 0.05 else "FLAG"
    print("  {:<34} DI {:.2f}   p {:.2f}   -> {}".format(name, di, p, verdict))
failures = gate(tests)
print()
print("CI gate: {}".format("PASS - safe to deploy" if not failures else "BLOCK - {} issue(s):".format(len(failures))))
for f in failures:
    print("   - " + f)
print("A green audit is the ship gate for a high-risk model (EU AI Act Art. 10). Re-run monthly - models update and biases drift.")

# Output:
# BIAS AUDIT REPORT  (model bump: classifier v2.3)
#   loan approval (after mitigation)   DI 0.85   p 0.21   -> OK
#   recommendation length by name      DI 0.88   p 0.04   -> FLAG
#   age in career suggestions          DI 0.93   p 0.42   -> OK
#
# CI gate: BLOCK - 1 issue(s):
#    - recommendation length by name: significant gap (p=0.04)
# A green audit is the ship gate for a high-risk model (EU AI Act Art. 10). Re-run monthly - models update and biases drift.

**Explanation**

This cell turns all the prior measurements into a standing gate: a small rules engine that scans aggregated test results and returns a blocking failure list, exactly the way a failing unit test blocks a merge. No model call - it operates on pre-collected audit numbers.

**How the code works - step by step**
- **`DI_FLOOR`** - the four-fifths threshold (0.80) the gate enforces.
- **`tests`** - a list of `(name, disparate_impact, p_value)` triples, the aggregated audit results.
- **`gate`** - for each test, flags a failure if disparate impact is below the floor OR the p-value is below 0.05, and collects all failures.
- The report loop prints each test's DI, p-value, and an OK/FLAG verdict.
- The final block prints the overall CI verdict (PASS or BLOCK) and lists every blocking issue.

**In one line:** aggregate the tests -> block the deploy on any four-fifths or significance failure -> re-run monthly.

**What you'll see:** A three-line audit report where the recommendation-length test is FLAGged (DI 0.88 but p 0.04), so the CI gate prints BLOCK - 1 issue(s) naming that significant gap, while the other two tests pass. The closing line ties it to the EU AI Act ship gate and monthly cadence.

Read as one loop, these seven cells are the whole discipline of measurable fairness: measure outcomes per group (selection rate, four-fifths, paired-prompt gaps), choose a fairness definition knowing they disagree and cannot all hold at once, prove the gap clears sampling noise, mitigate with the accuracy trade-off written down, then gate the audit so a biased model cannot ship silently. Every metric here is plain arithmetic on small integer tables - no scipy, no model call, no API key - which is exactly the point: bias detection is a check you can put in continuous integration and re-run monthly. From here, Lesson 15.2 answers *why* a model decided as it did (explainability), and the CI-gate pattern carries straight into the governance program that closes the module.